# Q4: Contraction Mapping Theorem

**Source:** Silver Lecture 2 (Bellman Equations)

This notebook:
1. Numerically verifies the contraction property of T^π and T*
2. Plots the convergence rate bound: $\|V_k - V^*\|_\infty \leq \gamma^k \|V_0 - V^*\|_\infty$
3. Demonstrates the theorem visually

See `src/q4_contraction/proof.md` for the formal mathematical proof.

In [ ]:
import sys

sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

from src.q1_student_mdp.mdp import (
    N_STATES,
    value_iteration,
)
from src.q4_contraction.demo import (
    bellman_optimality,
    verify_contraction,
)

## Theorem Statement

**Banach Fixed-Point Theorem:** If $(X, d)$ is a complete metric space and $T: X \to X$ satisfies
$$d(Tx, Ty) \leq \gamma \cdot d(x, y), \quad \gamma \in [0,1)$$

then $T$ has a **unique fixed point** $x^*$ and iterates $x_{k+1} = Tx_k$ converge with rate $O(\gamma^k)$.

**Application:** We show $T^\pi$ and $T^*$ are $\gamma$-contractions under $\|\cdot\|_\infty$.

## Numerical Verification of Contraction

In [ ]:
print('Verifying contraction property for multiple values of γ:')
print('=' * 60)
for g in [0.5, 0.9, 0.99]:
    verify_contraction(gamma=g, n_trials=500)
    print()

## Convergence Rate Analysis

Starting from $V_0 = 0$, the error satisfies:
$$\|V_k - V^*\|_\infty \leq \gamma^k \|V_0 - V^*\|_\infty$$

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, gamma in zip(axes, [0.5, 0.9, 0.99]):
    V_star = value_iteration(gamma=gamma)
    V = np.zeros(N_STATES, dtype=np.float64)
    err0 = float(np.max(np.abs(V - V_star)))

    errors, bounds, ks = [], [], []
    for k in range(1, 200):
        V = bellman_optimality(V, gamma)
        err = float(np.max(np.abs(V - V_star)))
        bound = (gamma ** k) * err0
        errors.append(err)
        bounds.append(bound)
        ks.append(k)
        if err < 1e-9:
            break

    ax.semilogy(ks, errors, lw=2, label=r'$\|V_k - V^*\|_\infty$', color='coral')
    ax.semilogy(ks, bounds, lw=2, linestyle='--',
                label=r'$\gamma^k \|V_0 - V^*\|_\infty$', color='steelblue')
    ax.set_xlabel('Iteration k')
    ax.set_ylabel('Error')
    ax.set_title(f'γ = {gamma}')
    ax.legend(fontsize=8)
    ax.grid(True, which='both', alpha=0.3)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

plt.suptitle('Value Iteration Convergence vs. Contraction Bound', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('q4_contraction_convergence.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: q4_contraction_convergence.png')

## Proof Summary (from `proof.md`)

**Key steps:**
1. $(\mathcal{V}, \|\cdot\|_\infty)$ is a **complete** metric space.
2. $|(T^\pi V_1)(s) - (T^\pi V_2)(s)| \leq \gamma \|V_1 - V_2\|_\infty$ (proven by triangle inequality + stochastic matrix row sums to 1).
3. $|(T^* V_1)(s) - (T^* V_2)(s)| \leq \gamma \|V_1 - V_2\|_\infty$ (proven via $|\max f - \max g| \leq \max|f-g|$).
4. Banach theorem → unique fixed points $V^\pi$ and $V^*$ with convergence rate $O(\gamma^k)$.
5. Policy iteration is **monotone** (V^{π'} ≥ V^π) + **finite** → must converge to V*.

**Why does the greedy algorithm converge?**
- Each policy improvement step strictly increases V (or stays equal at optimum)
- Since there are finitely many policies, it must terminate
- At termination, the greedy policy satisfies the Bellman optimality equation → it is π*

In [ ]:
# Demonstrate: faster γ → faster convergence
print('Iterations to convergence (error < 1e-6):')
for gamma in [0.5, 0.7, 0.9, 0.95, 0.99]:
    V_star = value_iteration(gamma=gamma)
    V = np.zeros(N_STATES, dtype=np.float64)
    k = 0
    while float(np.max(np.abs(V - V_star))) > 1e-6:
        V = bellman_optimality(V, gamma)
        k += 1
    print(f'  γ = {gamma:.2f}:  {k:4d} iterations   (theoretical bound: {int(np.log(1e-6/10) / np.log(gamma)) + 1})')